## Evaluation dei sistemi IR
Bisogna trovare un modo per misurare la qualità dei sistemi IR, in modo da poterli confrontare tra loro e migliorare quelli esistenti. Tuttavia, valutare un sistema IR significa **capire se gli utenti sono soddisfatti dei risultati** che ottengono quando fanno una ricerca, e non è facile misurarlo direttamente. Esistono diversi indicatori indiretti in questo senso: 
- Se i risultati restituiti sono **rilevanti** per il bisogno informativo dell'utente
- (quindi) se l'utente **clicca molto** sui risultati restituiti (attenzione in questo caso però, potrebbero essere ingannati da titoli fuorvianti)
- se gli utenti comprano un prodotto dopo aver effettuato una ricerca (ad esempio su Amazon)
- se gli utenti in generale **tornano** a utilizzare il sistema IR in futuro, oppure se lo abbandonano subito

La soluzione più usata e ragionevole per valutare un sistema IR è, in generale, quella di **misurare quanto i risultati sono rilevanti**. Per misurare la rilevanza dei risultati, sono chiaramente necessari tre elementi fondamentali:
1. Una **collezione di documenti** (dataset) da cui estrarre i risultati
2. Un insieme di **query** che rappresentano i bisogni informativi degli utenti da testare
3. Un insieme di **giudizi di rilevanza** che indicano se i risultati restituiti per ogni query sono rilevanti o meno. Si noti che i giudizi di rilevanza sono spesso ottenuti da **lavoro umano** che quindi costano moltissimo (servono persone esperte per poter giudicare documenti di qualsiasi argomento), e non sono pienamente affidabili (un giudice potrebbe pensarla diversamente da un altro).

Questo insieme di elementi costituisce un **benchmark** per la valutazione del sistema IR.

Qui sotto una tabella con benchmark "storici" per la valutazione IR. NQrys è il numero di query, NDocs è il numero di documenti nella collezione, e Q-D RelAss è il numero di giudizi di rilevanza disponibili.

<img src="img/tab.png" width="500px">

Inizialmente, prima degli anni '90, dataset molto piccoli con pochi documenti e facili da gestire (si parla di MB). A partire da inizio '90 con l'avvento del web è nato **TREC** (Text REtrieval Conference), un benchmark più grande (dell'ordine di 2GB) nato per confrontare in modo standard i sistemi di IR. Oggi TREC si usa ancora, ma il numero di pagine web è cresciuto enormemente (si parla di centinaia di milioni di pagine) e quindi esistono benchmark ancora più grandi, dell'ordine di TB (come ClueWeb).

Fondamentale capire che **la rilevanza di un documento NON è rispetto alla query in sé, ma all'originale bisogno informativo dell'utente.** Es. Need == "devo pulire la piscina", query == "pool cleaner" -> un documento sarà rilevante solo se risolve il bisogno informativo, non semplicemente se contiene le parole "pool" e "cleaner". Se ad esempio un documento restituito ha pool e cleaner ma parla di un software per la pulizia dei dati, allora non è davvero rilevante per il bisogno informativo dell'utente, per questo servono giudizi di rilevanza umani per capirlo.

**Perché Valutare un sistema IR?** Per capire qual è il miglior sistema IR, e quindi rispondere a domande tipo:
- Qual è il miglior algoritmo?
- Qual è il miglior ranking (cosine similarity, etc..)
- Qual è il miglior preprocessing (stemming, stop word removal, etc..)
- Qual è il miglior weighting (TF-IDF, BM25, etc..)
- Dove è meglio tagliare la lista dei risultati restituiti (top 10, top 100, etc..)

### Costruire il benchmark: Gold Standard
Come detto, per valutare un sistema IR è necessario costruire un benchmark, ed in particolare quindi serve una **collezione etichettata** di documenti. Per crearla, si seguono i seguenti passi, data una collezione documentale:
1. **Genera delle query particolarmente rappresentative** dei bisogni tipici degli utenti
2. **Utilizza un sistema IR base per recuperare tanti documenti potenzialmente rilevanti** (che abbia quindi alta recall, anche a costo di bassa precision)
3. **Gli esperti umani decideranno, per ogni documento, se questo è rilevante o meno per la query** (giudizi di rilevanza). Le etichette così ottenute costituiscono il gold standard (o **ground truth**) per la valutazione del sistema IR. Si parla di gold standard perché costa un sacco fare sto passaggio.

Quindi creare il Gold Standard è costoso in quanto richiede moltissimo lavoro umano (si deve infatti valutare una collezione documentale rappresentativa e quindi molto grande), ma è necessario al fine di valutare in modo accurato un sistema IR. Esistono anche metodi più economici per creare un benchmark, ad esempio utilizzando i click degli utenti come giudizi di rilevanza, ne parliamo alla fine del md. 

NB. Tipicamente decisione binaria (rilevante o non rilevante), ma si potrebbero anche usare giudizi di rilevanza più sfumati (ad esempio da 0 a 3, dove 0 == non rilevante, 1 == poco rilevante, 2 == abbastanza rilevante, 3 == molto rilevante).

### Precision, Recall, F1-Score
Le metriche per eccellenza per valutare un sistema IR sono **precision** e **recall**. D'ora in poi consideriamo come rilevante un documento se è stato giudicato rilevante dagli esperti umani (gold standard).
- **Precision**: **Tra tutti i documenti restituiti dal sistema IR, quanti sono effettivamente rilevanti**? $$Precision = \frac{TP}{TP + FP}$$ dove TP == true positives (documenti restituiti e rilevanti), FP == false positives (documenti restituiti ma non rilevanti).
- **Recall**: **Tra tutti i documenti rilevanti, quanti sono stati effettivamente restituiti dal sistema IR?** $$Recall = \frac{TP}{TP + FN}$$ dove FN == false negatives (documenti non restituiti ma rilevanti).

Un modo per ricordare velocemente è che né precision né recall guardano ai true negatives (TN), ovvero ai documenti non restituiti e non rilevanti, e che Precision guarda solo ai "positivi" P alla fine.

<img src="img/prec_recall.png" width="400px">

Intuitivamente, avere alta Recall significa trovare quasi tutti i documenti rilevanti, mentre avere alta Precision significa che quasi tutti i documenti restituiti sono rilevanti. In generale, **c'è un trade-off tra precision e recall**: se restituisco pochi documenti, è più probabile che siano rilevanti (alta precision), ma rischio di non restituire molti documenti rilevanti (bassa recall). Se invece restituisco molti documenti, è più probabile che trovi tutti quelli rilevanti (alta recall), ma rischio di restituire anche molti non rilevanti (bassa precision).

Dipende anche dall'applicazione: es. se faccio un test che dice se una persona ha una malattia, è più importante avere alta recall (non voglio che alcun paziente con la malattia sia testato negativo), anche a costo di avere bassa precision (a costo di avere falsi positivi e quindi persone non malate che risultano positive al test). Se invece faccio un sistema di raccomandazione per un e-commerce, è più importante avere alta precision (voglio consigliare solo prodotti rilevanti), anche a costo di avere bassa recall (potrei non consigliare tutti i prodotti rilevanti, ma almeno quelli che consiglio sono rilevanti).

E l'**Accuracy**? $$Accuracy = \frac{TP + TN}{TP + FP + FN + TN}$$ intuitivamente rappresenta la percentuale di documenti correttamente classificati (sia rilevanti che non rilevanti) rispetto al totale dei documenti. **Tuttavia in contesti come IR non è molto utile in quanto la collezione è enorme e quindi i documenti non rilevanti (TN) sono tantissimi e domina il calcolo**. Quindi anche se il sistema IR restituisse pochissimi documenti rilevanti (TP), il numero di TN sarebbe così alto da far sembrare il sistema IR molto accurato.

Allo stesso modo l'**Error Rate** (percentuale di documenti classificati in modo errato) non è utile in quanto il TN al denominatore porta sempre ad avere un error rate molto basso, anche se il sistema IR è pessimo.
$$Error Rate = \frac{FP + FN}{TP + FP + FN + TN}$$

Torniamo al trade-off tra precision e recall:
- **avere alta precision e bassa recall** significa restituire pochi documenti, ma quasi tutti rilevanti -> **perdo documenti utili**
- **avere alta recall e bassa precision** significa restituire molti documenti, ma molti non rilevanti -> **perdo tempo a leggere documenti inutili**

<img src="img/prec_recall_to.png" width="400px">

L'ideale sarebbe avere sia Precision che Recall ad 1, ma è praticamente impossibile e quindi nella pratica si cerca di trovare un compromesso tra i due. Per questo motivo, spesso si usa una metrica che combina precision e recall, come ad esempio l'**F1-Score**, che è la media armonica tra precision e recall: $$\text{F1-Score} = \frac{2}{\frac{1}{P} + \frac{1}{R}} = \frac{2 P R}{P + R}$$

L'uso della media armonica ($H = \frac{n}{\frac{1}{x_1} + \dots + \frac{1}{x_n}}$) invece della media aritmetica è giustificato dal fatto che l'armonica penalizza maggiormente valori sbilanciati: se si ha altissima precision ma bassa recall (o viceversa), si vuole comunque avere un F1-Score basso (mentre se avessi usato la media aritmetica avrei avuto un valore più alto). Dal grafico seguente si vede come l'armonica sia la metrica più severa rispetto alle altre, ad eccezione del minimo.

<img src="img/f1.png" width="400px">

L'esempio successivo di calcoli di precision e recall mostra inoltre come, nel restituire i documenti, i valori di precision e recall non sono fissi! Dipende da che tipo di documento è stato restituito. Quindi man mano che si restituiscono documenti la precision e la recall cambiano in base all'ordine, per poi stabilizzarsi solo una volta restituiti tutti i documenti.

<img src="img/bibo.png" width="400px">

### Rank-Based Measures
Dobbiamo ora definire delle metriche che **prendano in considerazione anche l'ordine dei documenti restituiti**. Infatti nei motori di ricerca l'ordine dei risultati è fondamentale, in quanto gli utenti tendono a cliccare principalmente sui primi risultati restituiti -> è importante assicurarsi che i primi risultati restituiti siano quelli più rilevanti.

Esistono in generale due modi per rappresentare la rilevanza di un documento:
- **Relevance binaria**: per ogni documento restituito, si indica se è rilevante o non rilevante (1 o 0), senza sfumature
- **Relevance graduata**: per ogni documento restituito, si indica un punteggio di rilevanza (ad esempio da 0 a 3, dove 0 == non rilevante, 1 == poco rilevante, 2 == abbastanza rilevante, 3 == molto rilevante)

Nel caso di Relevance binaria si usano metriche come **Precision@k** (precision calcolata sui primi k documenti restituiti) e **Recall@k** (recall calcolata sui primi k documenti restituiti), o **MAP**. Nel caso di Relevance graduata si usano metriche come **NDCG@k** (Normalized Discounted Cumulative Gain), che tiene conto sia del grado di rilevanza che dell'ordine dei documenti restituiti. 

(Intuitivamente, con la binaria si dice se un documento serve o meno, mentre con la graduata si dice quanto un documento serve)

#### Precision@k e Recall@k
Per **Precision@k** si calcola la precision considerando solo i primi k documenti restituiti: $$Precision@k = \frac{TP@k}{TP@k + FP@k} = \frac{\text{numero di rilevanti nei primi K}}{K}$$ 
dove TP@k == true positives tra i primi k documenti restituiti, FP@k == false positives tra i primi k documenti restituiti.

Precision@k risponde quindi alla domanda: **Tra i primi k documenti restituiti, quanti sono effettivamente rilevanti?** Si ignorano quindi del tutto i documenti restituiti dopo il k-esimo. 

<img src="img/prec_k.png" width="300px">

Si può definire in modo analogo **Recall@k**: $$Recall@k = \frac{TP@k}{TP@k + FN} = \frac{\text{numero di rilevanti nei primi K}}{\text{numero totale di rilevanti}}$$

Ha senso usare queste metriche proprio per guardare solo ai primi K risultati restituiti dal sistema IR, in quanto è lì che gli utenti tendono a cliccare. Se ad esempio un sistema IR restituisce 100 documenti, ma gli utenti cliccano solo sui primi 10, allora è più importante avere alta precision e recall nei primi 10 documenti restituiti, piuttosto che nei successivi 90.

A partire da queste metriche è possibile costruire il **Recall-Precision Graph**:

<img src="img/grrr.png" width="500px">

Il grafico è calcolato semplicemente come segue: data una query, si scorre il ranking dei documenti restituito. Per ogni documento restituito, si calcolano precision e recall considerando solo i documenti restituiti fino a quel punto, per questo si usano Precision@k e Recall@k.

Tuttavia, **per ogni query si ottiene una curva diversa, bisogna mediare**. 

Prima di farlo però è conveniente **interpolare** la curva, in quanto tipicamente la curva di precision recall non è monotona. Questo avviene perché ad esempio quando si trova un documento irrilevante, la precision cala, ma se se ne trovano poi dei rilevanti essa risale. La recall invece è sempre crescente, in quanto il numero totale di documenti rilevanti (denominatore) è fisso (mentre per precision il denominatore dipende da FP che varia in base a quello che estraggo). 

Per fare interpolazione e trasformare la curva irregolare in una monotona, si applica la formula $$P(R) = \max\{\, P' : R' \ge R \wedge (R',P') \in S \,\}$$ dove S è l'insieme dei punti osservati (R, P). Si sta cioè dicendo che la precision interpolata a un certo livello di recall R è data dalla massima precision osservata in qualsiasi punto con recall maggiore o uguale a R.

In pratica con l'interpolazione, se scelgo un valore di recall ad esempio R = 0.4, guardo tutti i punti nella curva con almeno 0.4 di recall e prendo la precision massima tra quei punti. Facendo così ottengo una curva monotona decrescente a gradini.

<img src="img/interp.png" width="300px">

Una volta ottenuta la curva interpolata per ogni query, si può fare la media tra tutte le curve per ottenere una curva che rappresenta la performance del sistema IR su tutte le query. La curva ha sull'asse x la recall e sull'asse y la precision media -> **se medio su 50 query la curva finale mi dirà il comportamento medio del sistema IR su 50 query**

<img src ="img/interp2.png" width="200px">

A questo punto è possibile confrontare diversi sistemi IR guardando le loro curve interpolated precision-recall: **il sistema IR migliore è quello che ha la curva più alta** (quindi **con precision più alta a parità di recall**, nell'esempio Stem).

<img src="img/comp.png" width="300px">

Il **breakeven point** della curva è il punto per cui **Precision == Recall**. Questo punto è interessante in quanto rappresenta un compromesso di massimo equilibro tra le due. Oggi il breakeven point è usato meno, e come metriche oggi sono molto più usate MAP, MRR e NDCG.

<img src="img/bp.png" width="300px">

### MAP (Mean Average Precision)
L'idea dietro MAP è che non conta più solo sapere quanti documenti rilevanti sono stati restituiti, ma anche **in che posizione** sono stati restituiti nel ranking. Si premaino i sistemi che mettono i documenti rilevanti più in alto nel ranking, in quanto è più probabile che gli utenti clicchino sui primi risultati restituiti.

Anzitutto si calcola l'**Average Precision (AP)**, data una query:
- si scorre il ranking dei documenti restituiti
- ogni volta che si trova un documento rilevante, si calcola la precision fino a quel punto (Precision@k)
- si fa la media di tutte le precision calcolate nei punti in cui si è trovato un documento rilevante

$$
AP = \frac{1}{R} \sum_{k \in \mathcal{R}} P@k
$$ 
dove R è il numero di documenti rilevanti, P@k è la precision calcolata sui primi k documenti restituiti. Si vede di seguito un esempio applicativo del calcolo di AP

<img src="img/ap.png" width="400px">

Chiaramente se i documenti rilevanti sono tutti in alto nel ranking, allora si ottiene un AP più alto, mentre se i documenti rilevanti sono sparsi in basso -> AP più basso. Infatti il ranking #1 è sicuramente migliore del secondo nell'esempio.

Per dare una misura quindi del motore di ricerca si calcola AP per più query selezionate, per poi fare la loro media ottenendo la **Mean Average Precision (MAP)**: $$MAP = \frac{1}{Q} \sum_{q=1}^{Q} AP_q$$ dove Q è il numero di query, AP_q è l'Average Precision calcolata per la query q. MAP rappresenta quindi la performance media del sistema IR su tutte le query testate, tenendo conto sia della rilevanza dei documenti restituiti che della loro posizione nel ranking.

<img src="img/map.png" width="400px">

Di seguito alcune importanti proprietà della MAP:
- MAP è compresa tra 0 e 1, dove 1 indica che tutti i documenti rilevanti sono stati restituiti e posizionati in cima al ranking per ogni query, mentre 0 indica che per ogni query nessun documento rilevante è mai stato restituito
- MAP è una **macro averaging**: ogni query ha lo stesso peso rispetto alle altre

**Il principale limite di MAP è che quindi assume che l'utente voglia trovare molti documenti rilevanti** per una query, quando in realtà tipicamente **gli utenti sono interessati solo al primo risultato**

Inoltre MAP richiede di sapere quali documenti sono rilevanti per ogni query, e quindi richiede tantt relevance judgments (etichette) (infatti MAP considera tutta la ranking list, non solo i primi risultati, e soprattutto si deve sapere esattamente quali sono di questi i documenti rilevanti)

### DCG (Discounted Cumulative Gain) e NDCG
Ora entriamo nel mondo delle metriche con **relevance graduata**: finora solo rilevanza 0 o 1 (rilevante o non rilevante), ma avrebbe senso definire misure per cui la rilevanza abbia più livelli (es. 0 inutile, 1 poco utile, 2 abbastanza utile, 3 molto utile).

La misura che introduciamo in questo senso è **DCG** (Discounted Cumulative Gain): **ci si chiede quanto valore (gain) si ottiene dai documenti restituiti, tenendo conto sia del loro grado di rilevanza che della loro posizione nel ranking**. L'idea è che quindi si ha gain più alto se un documento è rilevante ed è posizionato in alto nel ranking.

Si definisce la cumulative gain (CG) come la somma dei punteggi di rilevanza dei documenti restituiti: $CG@n = \sum_{i=1}^{n} r_i$ dove $r_i$ è il grado di rilevanza del documento $i$ e $n$ è il numero di documenti restituiti nel ranking.

Si introduce quindi anche la posizione nel conteggio con la formula:
$$DCG@n = r_1 + r_2 / \log_2(2) + r_3 / \log_2(3) + \dots + r_n / \log_2(n) = r_1 + \sum_{i=2}^{n} \frac{r_i}{\log_2(i)}$$

Formula alternativa:
$$DCG@n = \sum_{i=1}^{n} \frac{2^{r_i} - 1}{\log_2(i + 1)}$$

L'idea è che quindi ogni posizione viene **"scontata"**: per la posizione 1 non c'è sconto (è il primo risultato restituito, se era un documento rilevante ha senso lasciare il punteggio pieno), per la posizione 2 si divide per $\log_2(2) = 1$ (quindi anche qui non c'è sconto), per la posizione 3 si divide per $\log_2(3) \approx 1.58$ (quindi anche se era un documento molto rilevante, poiché sta in basso il suo contributo è meno) etc.. 

In questo modo più si scende nel ranking meno vale il contributo dei documenti restituiti, anche se sono rilevanti. La somma cumulativa ci permette di dire quanto valore totale otteniamo dal ranking restituito per quella query.

es. se abbiamo un ranking con relevance [3, 2, 3, 0, 0, 1, 2, 2, 3, 0] allora il CG sarebbe la somma, 16. Usando invece DCG 3 + 2/1 + 3/1.58 + 0/2 + 0/2.32 + 1/2.58 + 2/3 + 2/3.17 + 3/3.32 + 0/3.58 = 11.95

**DCG ha un problema: da solo non è confrontabile correttamente tra query diverse**. Immaginiamo infatti di avere un sistema di IR ottimale. Se la query ha molti documenti rilevanti -> DCG molto alto. Se invece la query ha pochi documenti rilevanti -> DCG sarà più basso, quando vorremmo avere comunque un punteggio alto in quanto il sistema IR è ottimale.

Per risolvere il problema si deve normalizzare il DCG rispetto alle query, introduciamo quindi **NDCG**:
$$NDCG@n = \frac{DCG@n}{IDCG@n}$$
dove $DCG@n$ è il DCG calcolato per la query nel ranking fino alla posizione n mentre $IDCG@n$ è il DCG calcolato per la query nel ranking ideale.

Il **Ranking ideale** è quello per cui i documenti sono ordinati in modo decrescente in base al loro grado di rilevanza (quindi prima i documenti con rilevanza 3, poi quelli con rilevanza 2, etc..). In questo modo, se il sistema IR restituisce un ranking che è uguale al ranking ideale, allora NDCG sarà uguale a 1 (massimo), mentre se il sistema IR restituisce un ranking completamente diverso dal ranking ideale, allora NDCG sarà più basso.

Nell'esempio seguente abbiamo 4 documenti d1, d2, d3 e d4 con rilevanza rispettivamente 0, 1, 2, 2. Il ranking ideale è quindi d4, d3, d2, d1. Si osserva quindi come per la query 1 (ranking function 1) l'ordine restituito è ottimale quindi il dcg è uguale all'idcg (ground thruth) -> NDCG == 1. Per la query 2 al contrario si ottiene un NDCG inferiore.

<img src="img/ndcg_example.png" alt="Esempio NDCG" width="500"/>

### MRR (Mean Reciprocal Rank)
Immaginiamo ora uno scenario dove l'utente è interessato a trovare **un solo documento rilevante** per soddisfare il suo bisogno informativo, quindi tale che esiste **solo un documento rilevante per ogni query**. 

esempi di questo sono: known-item search (sto cercando un documento specifico, ad esempio un libro, e voglio sapere se è presente nella collezione), navigational query (sto cercando un sito web specifico, ad esempio il sito di un ristorante) e looking for a fact (sto cercando una sola risposta per una domanda es. "quanti anni ha il presidente degli stati uniti?").

In questo caso non ci interessa trovare tanti documenti rilevanti, ma **vedere quanto in alto appare il documento utile**. In questo senso la metrica più adatta è il **Mean Reciprocal Rank (MRR)**, che misura la posizione del primo documento rilevante restituito dal sistema IR per ogni query. 

Si definisce anzitutto il Reciprocal Rank (RR) per una singola query come $RR = \frac{1}{K}$ dove K è la posizione in cui si trova il primo documento rilevante. Ad esempio, se il primo documento rilevante è in prima posizione, allora RR = 1/1 = 1 (massimo), se è in seconda posizione RR = 1/2 = 0.5 etc..

Dopodiché si fa semplicemente la media dei RR su più query ottenendo l'MRR:
$$
MRR =  \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{K_q} = \frac{1}{Q} \sum_{q=1}^{Q} RR_q
$$ 
dove K_q è la poszione del primo documento rilevante per la query q e Q è il numero totale di query.

### Valutazione con comportamento degli utenti
La valutazione con giudizi di rilevanza umani, per determinare il Gold Standard, è:
- Molto costosa (servono esperti umani per giudicare documenti di qualsiasi argomento)
- Inconsistente (un giudice potrebbe pensarla diversamente da un altro, o ancora nel corso del tempo un giudice potrebbe cambiare idea)
- Diventano obsoleti (il web è in continua evoluzione, quindi i giudizi di rilevanza potrebbero non essere più validi dopo un po' di tempo)
- Non sono sempre realistici (i giudici giudicano rispetto alla query, ma non conoscono il bisogno informativo originale dell'utente)

Per migliorare l'etichettamento l'idea potrebbe essere quindi di **utilizzare direttamente gli utenti** per ottenere/aggiornare i giudizi di rilevanza.

In questo senso l'intuizione più semplice è quella di utilizzare i **click degli utenti**: se data una query un risultato viene cliccato molte volte tra i vari utenti -> probabilmente è più rilevante rispetto ad altri risultati restituiti per quella query.

Tuttavia vi è un problema molto importante legato a questa idea: il **Position Bias**. Si è verificato empiricamente che **gli utenti tendono a cliccare di più sui primi risultati restituiti, indipendentemente dalla loro reale rilevanza**. Questo è dovuto al fatto che molti utenti danno per scontato che il sistema IR funzioni correttamente e quindi si fidano dei primi risultati restituiti.

<img src="img/pos_bias.png" width="500px">

Nell'esempio si vede addirittura come anche mettendo il ranking al contrario gli utenti continuano a cliccare di più sui primi risultati restituiti -> **Clicks are informative but biased**.

#### Relative vs Absolute Ratings
Per via del bias non è possibile quindi dire, dato un ranking di results, che ad esempio Result 1 è rilevante, Result 2 non lo è, Result 3 è molto rilevante... non cioè posso assegnare un labeling diretto a ogni risultato preso singolarmente se sfrutto i click degli utenti, infatti:
- il click è biased dalla posizione del risultato restituito
- il click è biased dal titolo/snippet del sito
- il click è biased da fattori esterni (ad esempio dalle abitudini/fretta dell'utente)

Ciò che è possibile fare è invece sfruttare i clicks per dare **confronti relativi** tra i risultati restituiti per una query. Se ad esempio l'utente clicca Result3 invece di Result1, è ragionevole dire che è difficile concludere che Result1 > Result3, ma **è più plausibile che Result3 > Result1**. (quindi si ragiona di confronti tra documenti restituiti per quella query)
 
**NOTA che però ciò non implica necessariamente che Result3 in generale, a livello assoluto, sia rilevante per quella query! Significa solo che probabilmente Result3 è più rilevante rispetto a Result1**

Importante dire inoltre che i confronti relativi non devono avvenire su ranking prodotti dallo stesso algoritmo X, perché quei click sono già influenzati dall'ordine dato da X e il relativo bias. Se poi uso quindi quei click per dire che X è buono -> rischio di confermare il bias di X. 

**Come confrontare quindi due ranking tramite clicks?** Immaginiamo di avere due sistemi che restituiscono per la stessa query due ranking diversi, Ranking A e Ranking B. Si potrebbe pensare di mostrare A a un gruppo di utenti e B a un altro gruppo di utenti, per poi confrontare i click. Tuttavia questo comporta molti problemi: utenti diversi, momenti diversi, bias di posizione diversi etc...

Un confronto più controllato in questo senso prevede l'uso della tecnica **Interleaving**: dati due ranking A e B, si mescolano in una singola lista, **alternando però i risultati**. Una volta ottenuta la lista, si rimuovono i duplicati. Ad esempio, se A restituisce [d1, d2, d3] e B restituisce [d3, d4, d5], allora una lista interleaved potrebbe essere [d1 (A), d3 (A,B), d2 (A), d4 (B), d5 (B)]. Il fatto che si parta da un doc di A o B è casuale.

Si fa ciò in modo da ridurre il bias quanto più possibile. Una volta ottenuta la lista, si contano i click degli utenti su di essa e si confrontano i click sui risultati di A rispetto a quelli di B. **Se A ottiene più click rispetto a B, allora probabilmente A è migliore.**

Interleaving quindi migliore perché davanti alla stessa query, con lo stesso gruppo di utenti e stessa pagina, "equilibrando" quindi bias di posizione, titolo/snippet, fattori esterni etc..

I motori di ricerca attuali usano tipicamente la tecnica di **A/B testing** per verificare se un aggiornamento del sistema migliora le performance rispetto alla versione precedente. In questo caso si sceglie una piccola porzione degli utenti (es. 0.1%) per usare la nuova versione, mentre il resto usa la vecchia. Dopodiché si raccolgono statistiche come il CTR (click-through rate), dwell time (tempo di permanenza sulla pagina dopo il click), query reformulation etc... per capire se la nuova versione sia adatta.

Si ricordi di fare attenzione in quanto comunque il comportamento degli utenti è utile, ma con molti problemi al di là del bias:
- noisy: click casuali o poco informativi
- difficile da interpretare, non sempre un click implica soddisfazione dell'utente
- spam/bots
- alcune query rare potrebbero non avere abbastanza click per essere valutate correttamente

#### Incorporating user behavior into ranking
Finora abbiamo visto solo come utilizzare ranking per valutare un sistema IR, ma è possibile anche **utilizzare il comportamento degli utenti per migliorare il ranking**.

Per farlo si può:
- **Fare Feature Engineering**: ad esempio inserire nel ranking segnali come CTR, dwell time, skip behavior in funzioni di ranking preesistenti tipo BM25
- **Fare Learned Ranking**: usare i dati di comportamento degli utenti per addestrare modelli di ranking più complessi, ad esempio basati su machine learning o deep learning, che imparano a prevedere la rilevanza dei documenti in base a segnali di comportamento degli utenti.